[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/708san/CV_data/blob/main/notebooks/colab/generation_smoke_test.ipynb)

# SDXL・LoRAのColab動作確認

このNotebookはライブラリの画像生成機能を試すための確認用です。Google Colabで **T4 GPU** を選んでください。SDXL本体とLoRAを初回に取得するため、数GBの空き容量と時間が必要です。

LoRAは4種類から1つを選べます。モデルのライセンス表示・学習元データ・注意点を一覧で確認し、選んだ1件の重みだけ取得してください。画像生成の出力に正確な文字や事実を期待しないでください。

使用例はプロンプトの例です。自分の目的に合わせて、実行前に文章を自由に書き換えられます。Image-to-Imageには合成画像か、利用者がColabへ明示アップロードした画像を選べます。Colabへのアップロードは利用者自身が行い、ライブラリは画像を別サービスへ自動送信しません。個人情報や共有する権利のない画像は使わないでください。

diffusion extraからDiffusers・PyTorch・PEFTをまとめて導入します。PEFTはPyPI公開版の0.21.2以降を使います。古いTorchAOが事前に入った環境でも通常のLoRAを読み込めるよう、ライブラリがLoRA読み込み中だけTorchAO連携判定を一時的に無効化します。TorchAOの削除や既存パッケージの入れ替えは行いません。

ランタイムの種類でT4 GPUを選んでから「すべてのセルを実行」してください。依存関係の導入後に再起動を求められた場合は、再起動して残りのセルを実行します。

In [ ]:
%pip install 'cv-data-lesson[diffusion]==0.1.0a7'

In [ ]:
import torch
from IPython.display import display
from cvlesson import loras

if not torch.cuda.is_available():
    raise RuntimeError('GPUが有効ではありません。ランタイムのタイプを変更し、GPUを選択してください。')

gpu_name = torch.cuda.get_device_name(0)
print(f'GPU: {gpu_name}')
if 'T4' not in gpu_name:
    print('この環境はT4ではありません。実行結果には実際のGPU名を記録してください。')

models = loras.list_models()
for model in models:
    print(f"\n{model['id']} — {model['name_ja']}")
    print(f"  モデルのライセンス表示: {model['license']['name']}")
    print(f"  学習元: {model['training_dataset']['repository']}")
    print(f"  学習元の条件: {model['training_dataset']['license']}")
    print(f"  利用上の説明: {model['rights_note']}")
    print(f"  モデルの特性: {model['capabilities_ja']}")
    print(f"  使用例（一例）: {'、'.join(model['use_examples_ja'])}")
    print(f"  注意: {model['limitations_ja']}")

print('上の権利表示と注意を確認してから、次のセルを実行してください。')

## 1つのLoRAだけ選んで試す

下の`LORA_ID`を登録一覧から1つ選び、`TASK`を文章から生成するか画像を描き変えるか選びます。選んだLoRAだけをダウンロードして画像を1枚生成します。他のLoRAの重みは取得しません。

### 生成設定の初期値

ライブラリの既定値は25 steps、guidance scale 7.0、Image-to-Image strength 0.65、トリガー語の自動追加ありです。下のNotebookでは、比較しやすいよう同じ値を明示しています。幅・高さを指定しない場合はDiffusersの既定サイズを使います。T4でメモリ不足なら幅・高さを512へ下げてください。

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
from cvlesson import generation, loras

# 登録一覧から1件選びます。例: fluent_emoji_3d / watercolor_sdxl / pixel_art_sdxl / thomas_cole_landscape
LORA_ID = 'fluent_emoji_3d'
TASK = 'text-to-image'  # 'image-to-image' も選べます
PROMPT = '明るい色の太陽をモチーフにした、丸みのある3Dアイコン'

NUM_INFERENCE_STEPS = 25
GUIDANCE_SCALE = 7.0
WIDTH = 768
HEIGHT = 768
SEED = 42
NEGATIVE_PROMPT = None
IMAGE_STRENGTH = 0.65
USE_TRIGGER = True

registered = {model['id'] for model in loras.list_models()}
if LORA_ID not in registered:
    raise ValueError(f'登録一覧にないLoRAです: {LORA_ID}')
if TASK not in {'text-to-image', 'image-to-image'}:
    raise ValueError("TASKは'text-to-image'か'image-to-image'にしてください")

source_image = None
SOURCE_IMAGE_PATH = '/content/your-image.jpg'  # 左側「ファイル」欄からアップロードした画像のパス
if TASK == 'image-to-image':
    if not Path(SOURCE_IMAGE_PATH).is_file():
        raise FileNotFoundError('Colab左側の「ファイル」欄から画像をアップロードし、SOURCE_IMAGE_PATHをそのファイル名に合わせてください。')
    with Image.open(SOURCE_IMAGE_PATH) as opened:
        source_image = opened.convert('RGB')
    display(source_image)

generator = generation.load(task=TASK, lora=LORA_ID, device='cuda')
kwargs = {
    'num_inference_steps': NUM_INFERENCE_STEPS,
    'guidance_scale': GUIDANCE_SCALE,
    'seed': SEED,
    'width': WIDTH,
    'height': HEIGHT,
    'negative_prompt': NEGATIVE_PROMPT,
    'use_trigger': USE_TRIGGER,
}
if source_image is not None:
    kwargs.update(image=source_image, strength=IMAGE_STRENGTH)
print(f'読み込み中: {LORA_ID} ({TASK})')
generated = generator.generate(PROMPT, **kwargs)
result_dir = Path('/content/cvlesson_lora_smoke_results')
result_dir.mkdir(parents=True, exist_ok=True)
output_path = result_dir / f'{LORA_ID}_{TASK}.png'
generated.save(output_path)
display(generated)
print('保存先:', output_path, ' / 画像サイズ:', generated.size)
print('選んだLoRA 1件の生成確認: OK')

## 結果を確認する

- 選んだLoRAで画像が1枚生成されたか確認します。
- `LORA_ID`・prompt・設定を変えて出力を比べられます。
- 出力は試作品です。生成画像だけで現地の状態や安全性を判断せず、文字は別のアプリで正しく入れてください。
- このNotebookの実行結果はColab上の実環境テストです。リポジトリへ生成画像やモデル重みをアップロードしないでください。